# ISL Translator — Training on Google Colab (REAL data, no synthetic)

**Author:** Krishna Vishwakarma (Roll 4848)  
**Pipeline:** MediaPipe 63-d wrist-relative landmarks → RF / SVM / CNN / CNN-BiLSTM → TFLite

## How to run (exact steps)
1. Go to https://colab.research.google.com → Upload → select `ISL_Training_Colab.ipynb` (from `notebooks/`).
2. Runtime → Change runtime type → GPU (optional, faster for CNN-BiLSTM).
3. Runtime → Run all. Cell 1 installs deps; Cell 2 downloads the REAL ISL-Hindi set via kagglehub (48k) + extracts MediaPipe landmarks (or upload your `data/processed/*.npy` when prompted).
4. After training, files `isl_model.h5`, `isl_model.tflite`, `confusion_matrix.png`, `training_history.png` appear in Files → download them.
5. Place downloads: `isl_model.tflite` + `isl_model.h5` → repo `models/`; PNGs → `screenshots/`; then continue Phase 4 (Flutter).

In [ ]:
# Cell 1 — setup (Colab)
!pip -q install mediapipe==0.10.14 kagglehub scikit-learn matplotlib seaborn keras-tuner
import os, sys, hashlib, random
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
print('setup ok')

In [ ]:
# Cell 2 — get REAL data: (a) upload processed .npy if you have them, else (b) download + MediaPipe extract
import kagglehub, cv2
from google.colab import files as _files
DATA = Path('/content/data'); PROC = DATA/'processed'; PROC.mkdir(parents=True, exist_ok=True)
print('Choose: upload X_train.npy etc. if you ran src/preprocess.py locally, else auto-download proceeds.')
# auto-download ISL-Hindi (48k, 40x1200) — public, no login
cache = kagglehub.dataset_download('krishbhagat/indian-sign-language-hindi')
print('cache:', cache)
import glob
base = glob.glob(os.path.join(cache, '**/HindiSignImages48x48'), recursive=True)[0]
print('base:', base, '| classes:', len(os.listdir(base)))

In [ ]:
# Cell 3 — MediaPipe extract (solutions API works in Colab with mediapipe 0.10) + md5 dedup + 70/15/15 group split
import mediapipe as mp
mp_hands = mp.solutions.hands
def hand_vec(bgr):
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5) as h:
        r = h.process(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        if not r.multi_hand_landmarks: return np.zeros(63, np.float32)
        p = np.array([[l.x,l.y,l.z] for l in r.multi_hand_landmarks[0].landmark], np.float32); p -= p[:1]; return p.reshape(-1)
classes = sorted(os.listdir(base))
X, y, seen, dup = [], [], set(), 0
for ci, c in enumerate(classes):
    for f in sorted(os.listdir(os.path.join(base, c))):
        p = os.path.join(base, c, f)
        h = hashlib.md5(open(p,'rb').read()).hexdigest()
        if h in seen: dup += 1; continue  # dedup: EDA found 1410 groups / 46k dups
        seen.add(h)
        img = cv2.imread(p)
        if img is None: continue
        X.append(hand_vec(img)); y.append(ci)
X = np.array(X, np.float32); y = np.array(y, np.int64)
print(f'X={X.shape} dup_skipped={dup} classes={len(classes)}')
from sklearn.model_selection import train_test_split
tr, tmp = train_test_split(np.arange(len(y)), test_size=0.30, random_state=42, stratify=y)
va, te = train_test_split(tmp, test_size=0.50, random_state=42, stratify=y[tmp])
np.save(PROC/'X_train.npy', X[tr]); np.save(PROC/'y_train.npy', y[tr])
np.save(PROC/'X_val.npy', X[va]); np.save(PROC/'y_val.npy', y[va])
np.save(PROC/'X_test.npy', X[te]); np.save(PROC/'y_test.npy', y[te])
open(PROC/'labels.txt','w').write('\n'.join(classes))
print(f'train={len(tr)} val={len(va)} test={len(te)}')

In [ ]:
# Cell 4 — Baseline 1: Random Forest + GridSearch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
Xtr, Xva, Xte = np.load(PROC/'X_train.npy'), np.load(PROC/'X_val.npy'), np.load(PROC/'X_test.npy')
ytr, yva, yte = np.load(PROC/'y_train.npy'), np.load(PROC/'y_val.npy'), np.load(PROC/'y_test.npy')
print(Xtr.shape, Xte.shape, len(np.unique(ytr)))
gs = GridSearchCV(RandomForestClassifier(random_state=42), {'n_estimators':[100,200],'max_depth':[None,20]}, cv=3, n_jobs=-1)
gs.fit(Xtr, ytr)
rf = gs.best_estimator_; print('best:', gs.best_params_)
pr = rf.predict(Xte)
acc = accuracy_score(yte, pr); p,r,f,_ = precision_recall_fscore_support(yte, pr, average='weighted', zero_division=0)
print(f'RF test — acc={acc:.4f} prec={p:.4f} rec={r:.4f} f1={f:.4f}')

In [ ]:
# Cell 5 — Baseline 2: SVM (RBF) — subsample if >8k for speed
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
Xtrs, Xtes = sc.fit_transform(Xtr), sc.transform(Xte)
n = min(8000, len(Xtrs)); idx = np.random.RandomState(42).choice(len(Xtrs), n, replace=False)
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(Xtrs[idx], ytr[idx])
ps = svm.predict(Xtes)
acc = accuracy_score(yte, ps); p,r,f,_ = precision_recall_fscore_support(yte, ps, average='weighted', zero_division=0)
print(f'SVM test — acc={acc:.4f} prec={p:.4f} rec={r:.4f} f1={f:.4f}')

In [ ]:
# Cell 6 — Baseline 3: Simple CNN (MLP on 63-d; reshape 21x3x1 for conv)
import tensorflow as tf
from tensorflow import keras
nc = len(open(PROC/'labels.txt').read().splitlines())
def reshape_cnn(X): return X.reshape(-1, 21, 3, 1)
cnn = keras.Sequential([keras.layers.Input((21,3,1)), keras.layers.Conv2D(32,3,activation='relu',padding='same'), keras.layers.MaxPool2D((2,1)), keras.layers.Flatten(), keras.layers.Dense(128,activation='relu'), keras.layers.Dropout(0.3), keras.layers.Dense(nc,activation='softmax')])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h1 = cnn.fit(reshape_cnn(Xtr), ytr, validation_data=(reshape_cnn(Xva), yva), epochs=20, batch_size=64, verbose=2)
pc = cnn.predict(reshape_cnn(Xte)).argmax(1)
acc = accuracy_score(yte, pc); p,r,f,_ = precision_recall_fscore_support(yte, pc, average='weighted', zero_division=0)
print(f'CNN test — acc={acc:.4f} prec={p:.4f} rec={r:.4f} f1={f:.4f}')

In [ ]:
# Cell 7 — Final: CNN-BiLSTM + KerasTuner search
import keras_tuner as kt
def build(hp):
    m = keras.Sequential([keras.layers.Input((21,3,1)), keras.layers.Conv2D(hp.Int('filters',16,64,step=16),3,activation='relu',padding='same'), keras.layers.MaxPool2D((2,1))])
    m.add(keras.layers.Reshape((10, -1)))
    m.add(keras.layers.Bidirectional(keras.layers.LSTM(hp.Int('lstm',32,128,step=32))))
    m.add(keras.layers.Dropout(hp.Float('drop',0.2,0.5,step=0.1)))
    m.add(keras.layers.Dense(nc, activation='softmax'))
    m.compile(optimizer=keras.optimizers.Adam(hp.Float('lr',1e-4,1e-2,sampling='log')), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m
tuner = kt.RandomSearch(build, objective='val_accuracy', max_trials=5, directory='/tmp/kt', project_name='isl', overwrite=True)
tuner.search(reshape_cnn(Xtr), ytr, validation_data=(reshape_cnn(Xva), yva), epochs=15, batch_size=64, verbose=2)
best = tuner.get_best_models(1)[0]; print('best hp:', tuner.get_best_hyperparameters(1)[0].values)
h2 = best.fit(reshape_cnn(Xtr), ytr, validation_data=(reshape_cnn(Xva), yva), epochs=25, batch_size=64, verbose=2)
final_model = best

In [ ]:
# Cell 8 — Evaluate ALL + confusion matrix + history plots (saved to screenshots/)
from sklearn.metrics import confusion_matrix
import pandas as pd
pf = final_model.predict(reshape_cnn(Xte)).argmax(1)
rows = []
for name, pred in [('RF',pr),('SVM',ps),('CNN',pc),('CNN-BiLSTM',pf)]:
    a = accuracy_score(yte, pred); p,r,f,_ = precision_recall_fscore_support(yte, pred, average='weighted', zero_division=0)
    rows.append([name, round(a,4), round(p,4), round(r,4), round(f,4)])
    print(f'{name}: acc={a:.4f} prec={p:.4f} rec={r:.4f} f1={f:.4f}')
display(pd.DataFrame(rows, columns=['model','accuracy','precision','recall','f1']))
cm = confusion_matrix(yte, pf)
plt.figure(figsize=(12,10)); sns.heatmap(cm, cmap='Blues'); plt.title('Confusion Matrix — CNN-BiLSTM (test)'); plt.xlabel('pred'); plt.ylabel('true'); plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150); plt.show()
plt.figure(figsize=(10,4)); plt.subplot(1,2,1); plt.plot(h2.history['accuracy'],label='train'); plt.plot(h2.history['val_accuracy'],label='val'); plt.title('accuracy'); plt.legend()
plt.subplot(1,2,2); plt.plot(h2.history['loss'],label='train'); plt.plot(h2.history['val_loss'],label='val'); plt.title('loss'); plt.legend(); plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150); plt.show()
print('saved confusion_matrix.png + training_history.png → move to screenshots/')

In [ ]:
# Cell 9 — Export final model (.h5 + .tflite) + latency check
import time
final_model.save('/content/isl_model.h5')
conv = tf.lite.TFLiteConverter.from_keras_model(final_model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
open('/content/isl_model.tflite','wb').write(conv.convert())
print('saved isl_model.h5 + isl_model.tflite')
# latency: single-sample inference
t0=time.time(); _ = final_model.predict(reshape_cnn(Xte[:1]), verbose=0); dt=(time.time()-t0)*1000
print(f'single-sample Keras latency: {dt:.1f} ms (target <500ms; TFLite on-device is faster)')
print('Download from Colab Files: isl_model.h5, isl_model.tflite, confusion_matrix.png, training_history.png')